In [1]:
from cs336_basics.blocks import TransformerLM

# Number of Parameters in Transformer
- token embedding: vd
- num_layers of TransfomerBlock, each of which has
    - one attention:
        - query/key/value projection: 3dd
        - output projection: dd
    - one feed forward: 3df
    - 2 rmsnorms, each of which has d params
- final rmsnorm: d
- lm head: dv
- total of above = vd+l(4dd+3df+2d)+d+dv

In [17]:
def compute_num_params(kwargs: dict) -> int:
    return (
        kwargs["vocab_size"] * kwargs["d_model"]
        + kwargs["num_layers"]
        * (
            4 * kwargs["d_model"] ** 2
            + 3 * kwargs["d_model"] * kwargs["d_ff"]
            + 2 * kwargs["d_model"]
        )
        + kwargs["d_model"]
        + kwargs["d_model"] * kwargs["vocab_size"]
    )

In [18]:
kwargs = dict(
    d_model=12,
    num_heads=3,
    d_ff=4,
    vocab_size=5,
    context_length=6,
    num_layers=7,
    theta=10_000,
)

In [19]:
model = TransformerLM(**kwargs)

In [20]:
sum(p.numel() for p in model.parameters())

5340

In [21]:
compute_num_params(kwargs)

5340

In [23]:
# 2B parameters, 8GB storage
compute_num_params(
    dict(
        d_model=1_600,
        num_heads=25,
        d_ff=6_400,
        vocab_size=50_257,
        context_length=1_024,
        num_layers=48,
        theta=10_000,
    )
) / 1e9

2.1270576

# Number of Flops in Transformer

- token embedding: no matmul
- num_layers of TransfomerBlock, each of which has
    - one attention:
        - query/key/value projection, each of which is bsdd
        - two rope, each of which is 2bsd
        - scaled dot product attention: 2bssd
        - output projection: bsdd
    - one feed forward: 3bsdf
    - two rmsnorms: no matmul
- final rmsnorm: no matmul
- lm head: bsdv
- total of above = l(4bsdd + 4bsd + 2bssd + 3bsdf) + bsdv ~ bsd(v+l(4d+2s+3f))
- multiply the sum of above by two, b/c each matmult takes 2mnp flops
- flops ~ 2bsd(v+l(4d+2s+3f))
- ignoring b, roughly speaking:
    - attention takes O(lssd)
    - query/key/value/output projection takes O(lsdd)
    - feed forward takes O(lsdf) ~ O(lsdd)
    - lm head takes O(sdv)
- number of flops: ffn > qkvo projection > attention > lm head

In [24]:
# assuming batch size = 1
def compute_num_flops(kwargs: dict) -> int:
    x = kwargs["vocab_size"] + kwargs["num_layers"] * (
        4 * kwargs["d_model"] + 2 * kwargs["context_length"] + 3 * kwargs["d_ff"]
    )
    return 2 * kwargs["context_length"] * kwargs["d_model"] * x

In [26]:
# 4T flops
compute_num_flops(
    dict(
        d_model=1_600,
        num_heads=25,
        d_ff=6_400,
        vocab_size=50_257,
        context_length=1_024,
        num_layers=48,
        theta=10_000,
    )
) / 1e12

4.5133365248

In [54]:
def compute_flops_breakdown(kwargs: dict) -> None:
    d_ff = kwargs["d_model"] * 4
    lm_head = kwargs["vocab_size"]
    ffn = kwargs["num_layers"] * d_ff * 3
    qkvo_proj = kwargs["num_layers"] * kwargs["d_model"] * 4
    attn = kwargs["num_layers"] * kwargs["context_length"] * 2

    total = lm_head + ffn + qkvo_proj + attn
    lm_head /= total
    ffn /= total
    qkvo_proj /= total
    attn /= total

    print(f"{ffn=:.2f}, {qkvo_proj=:.2f}, {attn=:.2f}, {lm_head=:.2f}")

In [55]:
common_kwargs = dict(
    vocab_size=50_257,
    context_length=1_024,
    theta=10_000,
)

In [56]:
compute_flops_breakdown(  # small
    dict(d_model=768, num_heads=12, num_layers=12, **common_kwargs)
)

ffn=0.50, qkvo_proj=0.17, attn=0.11, lm_head=0.23


In [57]:
compute_flops_breakdown(  # medium
    dict(d_model=1_024, num_heads=16, num_layers=24, **common_kwargs)
)

ffn=0.60, qkvo_proj=0.20, attn=0.10, lm_head=0.10


In [58]:
compute_flops_breakdown(  # large
    dict(d_model=1_280, num_heads=20, num_layers=36, **common_kwargs)
)

ffn=0.64, qkvo_proj=0.21, attn=0.09, lm_head=0.06


In [59]:
compute_flops_breakdown(  # XL
    dict(d_model=1_600, num_heads=25, num_layers=48, **common_kwargs)
)

ffn=0.67, qkvo_proj=0.22, attn=0.07, lm_head=0.04


As the model gets bigger, ffn and qkvo_proj take a larger proportion of flops; attn and lm_head take a smaller proportion of flops.
Reason: ffn and qkvo_proj are O(lsdd), attn is O(lssd), lm_head is O(sdv).
When model gets bigger, d gets bigger, so ffn and qkvo_proj scales quadratically.

In [61]:
# with a much longer context window, attn now takes a much larger proportion of flops
d = dict(d_model=1_600, num_heads=25, num_layers=48, **common_kwargs)
d.update({"context_length": 16_384})
compute_flops_breakdown(d)  # XL

ffn=0.32, qkvo_proj=0.11, attn=0.55, lm_head=0.02


In [63]:
# when we increase context length from 1k to 16k, flops increase from 4T to 150T
compute_num_flops(
    dict(
        d_model=1_600,
        num_heads=25,
        d_ff=6_400,
        vocab_size=50_257,
        context_length=16_384,
        num_layers=48,
        theta=10_000,
    )
) / 1e12

149.5227957248